In [155]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

data = sc.textFile("file:///home/itversity/employees.txt") 

In [ ]:
header = data.first()
data_no_header =data.filter(lambda line: line != header)

### Task 1
Parse text data into structured format

In [156]:
def clean_parse(line):
    parts = line.split(",")
    if len(parts) == 9 and parts[0].isdigit():
        try:
            return (int(parts[0]), parts[1], parts[2], parts[3], 
                    float(parts[4]), parts[5], parts[6], 
                    float(parts[7]), int(parts[8]))
        except ValueError:
            return None
    return None

structured_format = data.map(clean_parse).filter(lambda x: x is not None)
print(structured_format.take(1))

[(1, 'John Smith', 'Engineering', 'Senior Developer', 125000.0, 'San Francisco', '2021-03-15', 4.5, 8)]


### Task 2
Count occurrences of each name

In [157]:
name_counts = structured_format.map(lambda x: (x[1], 1)).reduceByKey(lambda a, b: a + b)

name_counts.collect()

[('Sarah Johnson', 1),
 ('Michael Williams', 1),
 ('Jennifer Brown', 1),
 ('David Jones', 1),
 ('Patricia Wilson', 1),
 ('James Anderson', 1),
 ('Mary Thomas', 1),
 ('Lisa Garcia', 1),
 ('John Smith', 1)]

### Task 3
Filter invalid records safely

In [158]:
invalid_records = data_no_header.filter(lambda line: clean_parse(line) is None)

print(invalid_records.collect())

['', '', '7,Robert Martinez,Legal,Legal Counsel145000,San Francisco,2019-09-22,4.8,10, 5', '']


### Task 4
Calculate the average salary per department

In [159]:
dept_salary = structured_format.map(lambda x: (x[2], (x[4], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
.mapValues(lambda x: x[0] / x[1])

dept_salary.collect()

[('Sales', 97500.0),
 ('Finance', 105000.0),
 ('IT', 115000.0),
 ('HR', 88000.0),
 ('Engineering', 121666.66666666667),
 ('Marketing', 92000.0)]

### Task 5
Calculate the number of employees for each department

In [160]:
dept_count = structured_format.map(lambda x: (x[2], 1)).reduceByKey(lambda a, b: a + b)

dept_count.collect()

[('Sales', 2),
 ('Finance', 1),
 ('Engineering', 3),
 ('Marketing', 1),
 ('IT', 1),
 ('HR', 1)]